# Molecule Enuermators

Application to enumerate given query molecules. 

The Custom Molecule Enumerator allows user to choose the site to enumerate with building blocks filtered based on given constraints. 

The Automated Molecule Enumerator, however, automates the substructure identification based on a reaction data and enumerates possible substructure compositions with building blocks filtered based on similarity to the substructures.

#### Helper Functions and imports

In [2]:
from enumerator import CustomEnumerator, AutomatedEnumerator
from rdkit.Chem import MolFromSmiles, MolToSmiles, MolFromSmarts, Draw, AllChem, AddHs, RemoveHs, rdFMCS

import ipywidgets as widgets
import nglview as nv
import utils

scaffold = MolFromSmiles('O=C(COS(=O)(NC)=O)NC1=NNC(C2=CC=CC=C2OCC)=C1')
# Example inputs
penicillin = 'CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C'
substructure = 'c1ccccc1CC(=O)'

In [3]:
## helper functions
# rgb color dict for rdkit
color_dict = {
    'blue': (0.19, 0.51, 0.70),
    'purple': (0.68, 0.45, 0.8),
    'pink': (0.94, 0.32, 0.65),
    'green': (0.48, 0.68, 0.35),
    'yellow': (0.81, 0.82, 0.0),
    'red': (0.95, 0.42, 0.19),
    'orange': (0.93, 0.69, 0.17)
}

# rule based constraints
rules = {
    'MW': (0, 500), # molecular weight
    'HBD': (0, 5), # hydrogen bond donors
    'HBA': (0, 10), # hydrogen bond acceptors
    'TPSA': (0, 200), # topological polar surface area
    'RotB': (0, 10), # rotatable bonds
    'Rings': (0, 10), # number of rings
    'ArRings': (0, 5), # number of aromatic rings
    'Chiral': (0, 5), # number of chiral centers
}

# reaction based constraints
reactions = utils.load_reactions_from_json('reactions/reactions.json')
reactions = [r for r in reactions if r.is_valid()]
reaction_tags = [r.get_tags() for r in reactions if r.is_valid()]
reaction_tags = list(set([tag for tags in reaction_tags for tag in tags]))

def get_nglview_mol(smi):
    mol = MolFromSmiles(smi)
    mol = AddHs(mol)
    AllChem.EmbedMolecule(mol)
    mol = RemoveHs(mol)
    return nv.show_rdkit(mol)

def text2svg(*args, fill: str = "#fff", font: str = "sans-serif", size: float = 16,
             baseline: float = 1, padding: float = 1, ratio: float = 1, text_anchor: str = "middle",
             background_fill: str = "firebrick", background_opacity: float = 1.0,
             width: float = 350, height: float = 100, 
             ):
    """
        Convert text to svg+xml format.
    """
    svg_out = f'<svg version="1.1" width="{width}" height="{height}" xmlns="http://www.w3.org/2000/svg">'
    svg_out += f'<rect width="80%" height="100%" fill="{background_fill}" opacity="{background_opacity}" x="10%" rx="20" ry="20" />'
    for i, text in enumerate(args):
        svg_out += f'<text x="{width//2}" y="{(height//2) + (i * size * ratio)}" font-family="{font}" font-size="{size}" text-anchor="{text_anchor}" fill="{fill}">{text}</text>'
    # svg_out += f'<text x="{width//2}" y="{height//2}" font-family="{font}" font-size="{size}" text-anchor="{text_anchor}" fill="{fill}">{text}</text>'
    svg_out += '</svg>'
    return svg_out.encode('utf-8')

def get_svg_mol(mol, sub_mol=None, sub_mol_color='green', legend=''):
    '''
        Get svg image of a molecule with a substructure highlighted.
    '''
    sub_mol_color = color_dict[sub_mol_color]
    if isinstance(mol, str):
       mol = MolFromSmiles(mol)
    AllChem.Compute2DCoords(mol)
    if sub_mol is not None:
        if isinstance(sub_mol, str):
            sub_struct = MolFromSmiles(sub_mol)
        else:
            sub_struct = sub_mol
        assert sub_struct is not None, 'Invalid substructure'
        assert mol.HasSubstructMatch(sub_struct), 'Substructure not found'
        hit_atoms = list(mol.GetSubstructMatch(sub_struct))
        hit_bonds = []
        for bond in sub_struct.GetBonds():
            a1 = hit_atoms[bond.GetBeginAtomIdx()]
            a2 = hit_atoms[bond.GetEndAtomIdx()]
            hit_bonds.append(mol.GetBondBetweenAtoms(a1, a2).GetIdx())
    else:
        hit_atoms = []
        hit_bonds = []
    drawing = Draw.MolDraw2DSVG(350, 100)
    drawing.DrawMolecule(mol, highlightAtoms=hit_atoms, highlightBonds=hit_bonds,
                          highlightAtomColors={i: sub_mol_color for i in hit_atoms},
                          highlightBondColors={i: sub_mol_color for i in hit_bonds},
                          legend=legend)
    drawing.FinishDrawing()
    svg = drawing.GetDrawingText()
    return svg.encode('utf-8')

def get_svg_mol_with_bbs(mol, bb1, bb2, bb_colors=['red', 'green'], legend=''):
    '''
        Similar to get_svg_mol, but instead of highlighting a substructure,
        it highlights the building blocks.
    '''
    bb_colors = [color_dict[c] for c in bb_colors]
    if isinstance(mol, str):
       mol = MolFromSmiles(mol)
    AllChem.Compute2DCoords(mol)
    if isinstance(bb1, str):
        bb1 = MolFromSmiles(bb1)
    if isinstance(bb2, str):
        bb2 = MolFromSmiles(bb2)
    assert bb1 is not None, 'Invalid building block 1'
    assert bb2 is not None, 'Invalid building block 2'
    
    mcs1 = rdFMCS.FindMCS([mol, bb1])
    mcs2 = rdFMCS.FindMCS([mol, bb2])
    smarts1 = mcs1.smartsString
    smarts2 = mcs2.smartsString
    bb1 = MolFromSmarts(smarts1)
    bb2 = MolFromSmarts(smarts2)
    hit_atoms1 = list(mol.GetSubstructMatch(bb1))
    hit_atoms2 = list(mol.GetSubstructMatch(bb2))
    hit_bonds1 = []
    hit_bonds2 = []
    for bond in bb1.GetBonds():
        a1 = hit_atoms1[bond.GetBeginAtomIdx()]
        a2 = hit_atoms1[bond.GetEndAtomIdx()]
        try:
            hit_bonds1.append(mol.GetBondBetweenAtoms(a1, a2).GetIdx())
        except:
            pass
    for bond in bb2.GetBonds():
        a1 = hit_atoms2[bond.GetBeginAtomIdx()]
        a2 = hit_atoms2[bond.GetEndAtomIdx()]
        try:
            hit_bonds2.append(mol.GetBondBetweenAtoms(a1, a2).GetIdx())
        except:
            pass
    drawing = Draw.MolDraw2DSVG(350, 150)
    drawing.DrawMolecule(mol, highlightAtoms=hit_atoms1+hit_atoms2, highlightBonds=hit_bonds1+hit_bonds2,
                          highlightAtomColors={**{i: bb_colors[0] for i in hit_atoms1}, **{i: bb_colors[1] for i in hit_atoms2}},
                          highlightBondColors={**{i: bb_colors[0] for i in hit_bonds1}, **{i: bb_colors[1] for i in hit_bonds2}},
                          legend=legend)
    drawing.FinishDrawing()
    svg = drawing.GetDrawingText()
    return svg.encode('utf-8')

def image_slider(mols, i):
    return get_svg_mol(MolToSmiles(mols[i]))

def custom_enumerator(molecule, building_blocks, method, substructure, rules, enumeration_kwargs):
    enumerator = CustomEnumerator(molecule=molecule, building_blocks=building_blocks, method=method)
    enumerator.add_substruct(substructure)
    enumerator.set_rules(**rules)
    enumerator.enumerate(**enumeration_kwargs)
    return enumerator

def automated_enumerator(molecule, building_blocks, sim_threshold, enumeration_kwargs):
    enumerator = AutomatedEnumerator(molecule=molecule, building_blocks=building_blocks, sim_threshold=sim_threshold)
    enumerator.enumerate(**enumeration_kwargs)
    return enumerator


## Custom Molecule Enumerator


In [6]:
## Custom Enumerator application using ipywidgets
# header
header = widgets.HTML(value="<h1>Custom Enumerator</h1>", layout=widgets.Layout(margin='0px 0px 0px 400px'))

# input widgets
molecule_input = widgets.Textarea(value='CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C', 
                                  placeholder='Type a molecule SMILES here', 
                                  description='Molecule:', layout=widgets.Layout(width='auto'))
building_block_source = widgets.Dropdown(options=['test', 'US_stocks', 'EU_stocks', 'global_stocks'], 
                                         layout=widgets.Layout(width='auto'), 
                                         description='BB source:')
method_choice = widgets.Dropdown(options=['rules', 'similarity', 'similarity_with_rules'],
                                 layout=widgets.Layout(width='auto'), 
                                 description='Method:')
subgraph_to_enumerate = widgets.Textarea(value='c1ccccc1CC(=O)', 
                                         placeholder='Substructure of the molecule to enumerate',
                                         description='Substructure:', layout=widgets.Layout(width='auto'))

# save file name
save_file_name = widgets.Text(value='enumerated_molecules.smi', 
                              placeholder='Enter file name to save enumerated molecules', 
                              description='Save as:', layout=widgets.Layout(width='auto'))

# rules as sliders
rule_sliders = {}
for rule, (min_val, max_val) in rules.items():
    if rule in ['MW', 'TPSA']:
        rule_sliders[rule] = widgets.FloatRangeSlider(value=[min_val, max_val], 
                                                      min=min_val, 
                                                      max=max_val, 
                                                      step=1, 
                                                      description=rule,
                                                      readout_format='.1f')
    else:
        rule_sliders[rule] = widgets.IntRangeSlider(value=[min_val, max_val], 
                                                    min=min_val, 
                                                    max=max_val, 
                                                    step=1, 
                                                    description=rule)
    rule_sliders[rule].layout.width = 'auto'
    rule_sliders[rule].layout.margin = '10px 0px 0px 0px'
    rule_sliders[rule].style.description_width = 'auto'
    rule_sliders[rule].style.handle_color = 'lightblue'

# buttons
update_button = widgets.Button(description='Update', 
                               layout=widgets.Layout(margin='0px 0px 0px 40px'),
                               style=widgets.ButtonStyle(button_color='lightblue'))
enumeration_button = widgets.Button(description='Enumerate',
                                    layout=widgets.Layout(margin='0px 0px 0px 40px'),
                                    style=widgets.ButtonStyle(button_color='orange'))
save_button = widgets.Button(description='Save',
                             layout=widgets.Layout(margin='0px 0px 0px 40px'),
                             style=widgets.ButtonStyle(button_color='lightgreen'))
button_box = widgets.HBox([update_button, enumeration_button, save_button], 
                          layout=widgets.Layout(margin='25px 0px 0px 0px', height='50px', width='600px'))

# molecule box
molecule_title = widgets.HTML(value='<h2>Molecule</h2>', 
                              layout=widgets.Layout(display='flex', margin='0px 0px 0px 210px'))
molecule_image = widgets.Image(value=get_svg_mol(molecule_input.value, subgraph_to_enumerate.value, sub_mol_color='red'), 
                               layout=widgets.Layout(display='flex', margin='-20px 0px 0px 0px'),
                               format='svg+xml',)
molecule_box = widgets.VBox([molecule_title, molecule_image],)

# enumeration box with slider to view resulting enumeration after clicking the button
enumeration_title = widgets.HTML(value='<h2>Enumeration Results</h2>', 
                                 layout=widgets.Layout(margin='0px 0px 0px 160px'))
enumeration_slider = widgets.IntSlider(layout=widgets.Layout(margin='0px 0px 0px 150px'),)
enumeration_image = widgets.Image(value=text2svg('Click Enumerate Button!', fill='black', background_fill='white'),
                                  layout=widgets.Layout(display='flex', margin='0px 0px 0px 0px'),
                                  format='svg+xml',)
enumeration_box = widgets.VBox([enumeration_title, enumeration_slider, enumeration_image], )

# events
def click_on_update(b):
    '''
        Update button event. It will update the molecule image after changing the 
        molecule or substructure.
    '''
    molecule_image.value = get_svg_mol(molecule_input.value, subgraph_to_enumerate.value, sub_mol_color='red')
update_button.on_click(click_on_update)

def click_on_enumerate(b):
    '''
        Enumerate button event. It will call the enumerator function for 
        the given inputs.
    '''
    molecule = molecule_input.value
    building_blocks = building_block_source.value
    method = method_choice.value
    substructure = subgraph_to_enumerate.value
    rules = {rule: rule_sliders[rule].value for rule in rule_sliders}
    enumeration_kwargs = {'include': [], 'exclude': [], 'sim_threshold': 0.1}
    enumerator = custom_enumerator(molecule, building_blocks, method, substructure, rules, enumeration_kwargs)
    if len(enumerator.enumerated_molecules) == 0:
        enumeration_image.value = text2svg('No Enumerations!', 'Check Rules and Substructure!')
        enumeration_slider.max = 0
        return
    enumeration_slider.max = len(enumerator.enumerated_molecules) - 1
    enumeration_slider.value = 0
    enumeration_slider.step = 1

    def on_value_change(change):
        enumeration_slider.value = change['new']
        enumeration_image.value = get_svg_mol(enumerator.enumerated_molecules[change['new']], enumerator._core_mol, sub_mol_color='green')
    enumeration_slider.observe(on_value_change, names='value')
    enumeration_image.value = get_svg_mol(enumerator.enumerated_molecules[0], enumerator._core_mol, sub_mol_color='green')

    def click_on_save(b):
        '''
            Save button event. It will save the enumerated molecules to a file.
        '''
        if len(enumerator.enumerated_molecules) == 0:
            enumeration_image.value = text2svg('No Enumerations!', "Can't Save!")
            return
        else:
            enumerator.save_enumerated_molecules('test.smi')
            enumeration_image.value = text2svg(f'Saved to {save_file_name.value}!', background_fill='green')
        enumerator.save_enumerated_molecules(save_file_name.value)
    save_button.on_click(click_on_save)

enumeration_button.on_click(click_on_enumerate)


# grid layout
grid = widgets.GridspecLayout(12, 4, height='600px', width='1100px', grid_gap='5px')
grid[0, :] = header
grid[1, :2] = molecule_input
grid[2, :2] = subgraph_to_enumerate
grid[3, :2] = building_block_source
grid[4, :2] = method_choice
grid[5, :2] = save_file_name
for i, (rule, slider) in enumerate(rule_sliders.items()):
    grid[(i%4)+6, i//4] = slider
grid[10, :2] = button_box
grid[1:5, 2:] = molecule_box
grid[5:, 2:] = enumeration_box

grid


GridspecLayout(children=(HTML(value='<h1>Custom Enumerator</h1>', layout=Layout(grid_area='widget001', margin=…

## Automated Molecule Enumerator


In [4]:
## Automated Enumerator application using ipywidgets
# header
header = widgets.HTML(value="<h1>Automated Enumerator</h1>", layout=widgets.Layout(margin='0px 0px 0px 400px'))

# input widgets
molecule_input = widgets.Textarea(value='CC1(C(N2C(S1)C(C2=O)NC(=O)CC3=CC=CC=C3)C(=O)O)C', 
                                  placeholder='Type a molecule SMILES here', 
                                  description='Molecule:', layout=widgets.Layout(width='auto'))
building_block_source = widgets.Dropdown(options=['test', 'US_stocks', 'EU_stocks', 'global_stocks'], 
                                         layout=widgets.Layout(width='auto'), 
                                         description='BB source:')
similarity_threshold = widgets.FloatSlider(value=0.15, 
                                           min=0.0, 
                                           max=1.0, 
                                           step=0.01, 
                                           description='Sim Cutoff:', 
                                           readout_format='.2f',
                                           style=widgets.SliderStyle(handle_color='lightblue'),
                                           layout=widgets.Layout(width='auto'))
rxn_tag_desc = widgets.HTML(value='Reaction Tags:',
                            layout=widgets.Layout(margin='0px 0px 0px 0px', width='150px'))
rxn_tags = widgets.TagsInput(value=['amide coupling', 'amide', 'C-N bond formation', 'C-N', 
                                    'alkylation', 'N-arylation', 'azole', 'amination'],
                             allowed_tags=reaction_tags,
                             allow_duplicates=False,
                             layout=widgets.Layout(width='auto'),)
rxn_inputs = widgets.HBox([rxn_tag_desc, rxn_tags], 
                         layout=widgets.Layout(margin='0px 0px 0px 0px', height='100px'))

# save file name
save_file_name = widgets.Text(value='enumerated_molecules.csv', 
                              placeholder='Enter file name to save enumerated molecules', 
                              description='Save as:', layout=widgets.Layout(width='auto'))

# buttons
update_button = widgets.Button(description='Update', 
                               layout=widgets.Layout(margin='0px 0px 0px 40px'),
                               style=widgets.ButtonStyle(button_color='lightblue'))
enumeration_button = widgets.Button(description='Enumerate',
                                    layout=widgets.Layout(margin='0px 0px 0px 40px'),
                                    style=widgets.ButtonStyle(button_color='orange'))
save_button = widgets.Button(description='Save',
                             layout=widgets.Layout(margin='0px 0px 0px 40px'),
                             style=widgets.ButtonStyle(button_color='lightgreen'))
button_box = widgets.HBox([update_button, enumeration_button, save_button], 
                          layout=widgets.Layout(margin='0px 0px 0px 0px', height='50px', width='600px'))

# molecule box
molecule_title = widgets.HTML(value='<h2>Molecule</h2>', 
                              layout=widgets.Layout(display='flex', margin='0px 0px 0px 210px'))
molecule_image = widgets.Image(value=get_svg_mol(molecule_input.value), 
                               layout=widgets.Layout(display='flex', margin='-20px 0px 0px 0px'),
                               format='svg+xml',)
molecule_box = widgets.VBox([molecule_title, molecule_image],)

# enumeration box with slider to view resulting enumeration after clicking the button
enumeration_title = widgets.HTML(value='<h2>Enumeration Results</h2>', 
                                 layout=widgets.Layout(margin='0px 0px 0px 160px'))
enumeration_slider = widgets.IntSlider(layout=widgets.Layout(margin='0px 0px 0px 150px'),)
enumeration_image = widgets.Image(value=text2svg('Click Enumerate Button!', fill='black', background_fill='white'),
                                  layout=widgets.Layout(display='flex', margin='0px 0px 0px 0px'),
                                  format='svg+xml',)
enumeration_box = widgets.VBox([enumeration_title, enumeration_slider, enumeration_image], 
                               layout=widgets.Layout(margin='0px 0px 0px 0px'))

# events
def click_on_update(b):
    '''
        Update button event. It will update the molecule image after changing the 
        molecule or substructure.
    '''
    molecule_image.value = get_svg_mol(molecule_input.value)
update_button.on_click(click_on_update)

def click_on_enumerate(b):
       '''
              Enumerate button event. It will call the enumerator function for 
              the given inputs.
       '''
       molecule = molecule_input.value
       building_blocks = building_block_source.value
       sim_threshold = similarity_threshold.value
       enumeration_kwargs = {'reaction_tags': rxn_tags.value, 'n_compositions': 5}
       enumerator = automated_enumerator(molecule, building_blocks, sim_threshold, enumeration_kwargs)
       if len(enumerator.enumerated_molecules) == 0:
              enumeration_image.value = text2svg('No Enumerations!', 'Check Constraints!')
              enumeration_slider.max = 0
              return
       enumeration_slider.max = len(enumerator.enumerated_molecules) - 1
       enumeration_slider.value = 0
       enumeration_slider.step = 1
       
       def on_value_change(change):
              enumeration_slider.value = change['new']
              idx = [i for i, r in enumerate(reactions) if r.name == enumerator.enumerated_molecules[change['new']][3]][0]
              reaction_site = reactions[idx].get_products()[-1]
              enumeration_image.value = get_svg_mol_with_bbs(*enumerator.enumerated_molecules[change['new']][:3],
                                                             bb_colors=['purple', 'green'],
                                                             legend=enumerator.enumerated_molecules[change['new']][3])
       enumeration_slider.observe(on_value_change, names='value')
       enumeration_image.value = get_svg_mol_with_bbs(*enumerator.enumerated_molecules[0][:3],
                                                      bb_colors=['purple', 'green'],
                                                      legend=enumerator.enumerated_molecules[0][3])
       # add save button event
       def click_on_save(b):
              '''
                     Save button event. It will save the enumerated molecules to a csv file.
                     Save existing enumerated molecules to a csv file.
              '''
              if len(enumerator.enumerated_molecules) == 0:
                     enumeration_image.value = text2svg('No Enumerations!', "Can't Save!")
                     return
              else:
                     enumerator.save_enumerated_molecules(f'{save_file_name.value}')
                     enumeration_image.value = text2svg(f'Saved to {save_file_name.value}!', background_fill='green')
       save_button.on_click(click_on_save)
enumeration_button.on_click(click_on_enumerate)

# grid layout
grid = widgets.GridspecLayout(12, 4, height='650px', width='1100px', grid_gap='5px')
grid[0, :] = header
grid[1, :2] = molecule_input
grid[2, :2] = building_block_source
grid[3, :2] = save_file_name
grid[4, :2] = similarity_threshold
grid[5, :2] = rxn_inputs
grid[9, :2] = button_box
grid[1:5, 2:] = molecule_box
grid[5:, 2:] = enumeration_box

grid


GridspecLayout(children=(HTML(value='<h1>Automated Enumerator</h1>', layout=Layout(grid_area='widget001', marg…